# Audit des rendements Fama–French et de la cible à cinq séances

> **English Overview (Step 1 — Data Provenance & Target Construction):** Audits the Fama–French daily factor archive (identified by its SHA-256) and constructs the 5-session forward annualized realized volatility target `RV5_t = sqrt((252/5) * sum_{k=1..5} r_{t+k}^2)` on 26,312 valid dates, with an explicit date alignment: the target at `t` uses only returns after `t`.

**Question unique :** pouvons-nous construire la volatilité réalisée des cinq séances futures sans ambiguïté d'unité, de date ou d'alignement temporel ?

Ce notebook importe uniquement les facteurs quotidiens officiels de Kenneth French. Il ne contient aucun modèle prédictif. La cible associée à la date $t$ utilise exclusivement les rendements des séances $t+1$ à $t+5$ :

$$RV_{5,t}=\sqrt{\frac{252}{5}\sum_{k=1}^{5}r_{t+k}^{2}}.$$

## 1. Importations et conventions

La bibliothèque publie `Mkt-RF` et `RF` en pourcentage. Le rendement total du marché est donc reconstruit par `Mkt-RF + RF`, puis divisé par 100 pour obtenir un rendement décimal.

In [1]:
from datetime import datetime, timezone
from hashlib import sha256
from io import BytesIO, StringIO
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import numpy as np
import pandas as pd

SOURCE_URL = (
    'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/'
    'F-F_Research_Data_Factors_daily_CSV.zip'
)
EXPECTED_SHA256 = '2f29e22546069914890a712680a6f81f3680ebc3543b52864484d209ca13a7db'
SNAPSHOT_FILENAME = (
    'F-F_Research_Data_Factors_daily_202608_'
    f'sha256-{EXPECTED_SHA256}.zip'
)
ANNUALIZATION_DAYS = 252
HORIZON = 5
print(f'pandas={pd.__version__}; numpy={np.__version__}')

pandas=2.2.3; numpy=2.2.1


## 2. Téléchargement et lecture contrôlée

Une copie brute immuable est conservée localement sous `data/raw/fama_french/` et exclue de Git. Le notebook la recherche depuis la racine du projet ou depuis le dossier `notebooks`. Si elle est absente, il télécharge la source officielle en mémoire. Dans les deux cas, l'empreinte SHA-256 doit correspondre exactement au millésime validé ; une source modifiée provoque un arrêt explicite.

In [2]:
loaded_at_utc = datetime.now(timezone.utc).isoformat()
snapshot_candidates = [
    Path('data/raw/fama_french') / SNAPSHOT_FILENAME,
    Path('../data/raw/fama_french') / SNAPSHOT_FILENAME,
]
local_snapshot = next((path for path in snapshot_candidates if path.is_file()), None)
if local_snapshot is not None:
    archive_bytes = local_snapshot.read_bytes()
    data_origin = f'copie locale immuable : {local_snapshot.as_posix()}'
else:
    with urlopen(SOURCE_URL, timeout=60) as response:
        archive_bytes = response.read()
    data_origin = 'téléchargement en mémoire depuis la source officielle'
archive_sha256 = sha256(archive_bytes).hexdigest()
assert archive_sha256 == EXPECTED_SHA256, (
    f'Empreinte inattendue : {archive_sha256}. ' 
    f'Empreinte attendue : {EXPECTED_SHA256}.'
)

with ZipFile(BytesIO(archive_bytes)) as archive:
    members = archive.namelist()
    if len(members) != 1:
        raise ValueError(f'Archive inattendue : {members}')
    member_name = members[0]
    text = archive.read(member_name).decode('utf-8-sig')

lines = text.splitlines()
header_index = next(i for i, line in enumerate(lines) if line.strip().startswith(',Mkt-RF'))
data_lines = [lines[header_index]]
for line in lines[header_index + 1:]:
    first_field = line.split(',', 1)[0].strip()
    if len(first_field) == 8 and first_field.isdigit():
        data_lines.append(line)
    elif data_lines:
        break

factors = pd.read_csv(StringIO('\n'.join(data_lines)))
factors = factors.rename(columns={factors.columns[0]: 'date'})
factors['date'] = pd.to_datetime(factors['date'].astype(str), format='%Y%m%d', errors='raise')
factors = factors.set_index('date')

print(f'URL : {SOURCE_URL}')
print(f'Origine des octets : {data_origin}')
print(f'Chargé à : {loaded_at_utc}')
print(f'Fichier : {member_name}; octets ZIP : {len(archive_bytes):,}')
print(f'SHA-256 : {archive_sha256}')
print(f'Note du producteur : {lines[0]}')
print(f'Observations lues : {len(factors):,}')

URL : https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_daily_CSV.zip
Origine des octets : copie locale immuable : data/raw/fama_french/F-F_Research_Data_Factors_daily_202608_sha256-2f29e22546069914890a712680a6f81f3680ebc3543b52864484d209ca13a7db.zip
Chargé à : 2026-10-15T00:34:18.185243+00:00
Fichier : F-F_Research_Data_Factors_daily.csv; octets ZIP : 178,184
SHA-256 : 2f29e22546069914890a712680a6f81f3680ebc3543b52864484d209ca13a7db
Note du producteur : This file was created by using the 202608 CRSP database.
Observations lues : 26,317


## 3. Vérification des unités et des dates

Les contrôles suivants échouent explicitement si les colonnes attendues, l'ordre des dates, l'unicité ou les unités deviennent incompatibles avec nos conventions. Le calendrier historique est conservé tel quel : il comprend notamment d'anciennes séances du samedi. Aucun jour civil absent n'est ajouté.

In [3]:
expected_columns = ['Mkt-RF', 'SMB', 'HML', 'RF']
assert factors.columns.tolist() == expected_columns, factors.columns.tolist()
assert factors.index.is_monotonic_increasing
assert not factors.index.has_duplicates
assert factors.notna().all().all()

# Les valeurs brutes sont des pourcentages journaliers, non des décimaux.
assert factors['Mkt-RF'].abs().quantile(0.99) > 0.1
assert factors['Mkt-RF'].abs().max() < 100
factors['market_return_pct'] = factors['Mkt-RF'] + factors['RF']
factors['market_return'] = factors['market_return_pct'] / 100.0
assert np.allclose(factors['market_return'] * 100, factors['market_return_pct'])

date_gaps = factors.index.to_series().diff().dt.days
print(f'Première date : {factors.index.min().date()}')
print(f'Dernière date : {factors.index.max().date()}')
print(f'Dates dupliquées : {factors.index.duplicated().sum()}')
print(f'Séances un samedi : {(factors.index.dayofweek == 5).sum()}')
print(f'Séances un dimanche : {(factors.index.dayofweek == 6).sum()}')
print(f'Valeurs manquantes brutes : {int(factors[expected_columns].isna().sum().sum())}')
print(f'Écart civil maximal entre deux séances : {int(date_gaps.max())} jours')
print('Exemple de conversion :')
print(factors[['Mkt-RF', 'RF', 'market_return_pct', 'market_return']].head(3).to_string())

Première date : 1926-07-01
Dernière date : 2026-08-31
Dates dupliquées : 0
Séances un samedi : 1158
Séances un dimanche : 0
Valeurs manquantes brutes : 0
Écart civil maximal entre deux séances : 12 jours
Exemple de conversion :
            Mkt-RF    RF  market_return_pct  market_return
date                                                      
1926-07-01    0.09  0.01               0.10         0.0010
1926-07-02    0.45  0.01               0.46         0.0046
1926-07-06    0.17  0.01               0.18         0.0018


## 4. Construction de la cible strictement future

Chaque colonne `r_t_plus_k` est créée par un décalage négatif de $k$ lignes : à la ligne datée $t$, elle contient le rendement de la $k$-ième séance suivante. `min_count=5` interdit une cible partielle en fin d'échantillon.

In [4]:
future_returns = pd.concat(
    {f'r_t_plus_{k}': factors['market_return'].shift(-k) for k in range(1, HORIZON + 1)},
    axis=1,
)
future_sum_squares = future_returns.pow(2).sum(axis=1, min_count=HORIZON)
factors['rv_5'] = np.sqrt((ANNUALIZATION_DAYS / HORIZON) * future_sum_squares)

assert factors['rv_5'].notna().sum() == len(factors) - HORIZON
assert factors['rv_5'].iloc[-HORIZON:].isna().all()
assert (factors['rv_5'].dropna() >= 0).all()
print(f'Cibles complètes : {factors.rv_5.notna().sum():,}')
print(f'Cibles manquantes : {factors.rv_5.isna().sum()} (les {HORIZON} dernières dates)')
print(f'Première cible valide : {factors.rv_5.first_valid_index().date()}')
print(f'Dernière cible valide : {factors.rv_5.last_valid_index().date()}')

Cibles complètes : 26,312
Cibles manquantes : 5 (les 5 dernières dates)
Première cible valide : 1926-07-01
Dernière cible valide : 2026-08-24


## 5. Contrôles manuels sur trois observations

Le calcul est répété directement à partir des cinq lignes suivantes pour une observation au début, au milieu et à la fin de l'échantillon utilisable. La différence attendue est nulle à l'arrondi numérique près.

In [5]:
positions = [0, len(factors) // 2, len(factors) - HORIZON - 1]
manual_rows = []
for position in positions:
    forecast_date = factors.index[position]
    target_slice = factors['market_return'].iloc[position + 1:position + 1 + HORIZON]
    manual_value = float(np.sqrt((ANNUALIZATION_DAYS / HORIZON) * np.square(target_slice).sum()))
    notebook_value = float(factors['rv_5'].iloc[position])
    manual_rows.append({
        'date_t': forecast_date.date().isoformat(),
        'dates_cible': ' → '.join(d.date().isoformat() for d in target_slice.index),
        'rv_5_notebook': notebook_value,
        'rv_5_manuel': manual_value,
        'difference': notebook_value - manual_value,
    })
manual_checks = pd.DataFrame(manual_rows)
assert np.allclose(manual_checks['difference'], 0.0, atol=1e-15)
print(manual_checks.to_string(index=False))

    date_t                                                    dates_cible  rv_5_notebook  rv_5_manuel  difference
1926-07-01 1926-07-02 → 1926-07-06 → 1926-07-07 → 1926-07-08 → 1926-07-09       0.063375     0.063375         0.0
1974-06-20 1974-06-21 → 1974-06-24 → 1974-06-25 → 1974-06-26 → 1974-06-27       0.190371     0.190371         0.0
2026-08-24 2026-08-25 → 2026-08-26 → 2026-08-27 → 2026-08-28 → 2026-08-31       0.065418     0.065418         0.0


## 6. Démonstration de l'alignement et de l'absence de fuite

La cible est nécessairement connue après $t$ ; elle ne doit donc jamais servir de variable explicative à $t$. Le contrôle ci-dessous établit mécaniquement que ses cinq dates constitutives sont toutes strictement postérieures à la date d'observation. L'absence de fuite d'un futur modèle devra être réauditée séparément pour ses variables explicatives.

In [6]:
alignment = pd.DataFrame(index=factors.index[:-HORIZON])
alignment.index.name = 'date_t'
for k in range(1, HORIZON + 1):
    alignment[f'date_t_plus_{k}'] = factors.index.to_series().shift(-k).iloc[:-HORIZON].to_numpy()
    assert (alignment[f'date_t_plus_{k}'] > alignment.index).all()

# Vérification indépendante : la matrice décalée correspond exactement aux lignes futures.
for position in positions:
    expected = factors['market_return'].iloc[position + 1:position + 1 + HORIZON].to_numpy()
    observed = future_returns.iloc[position].to_numpy()
    assert np.array_equal(observed, expected)

print('Tous les rendements constitutifs de RV_5 sont strictement postérieurs à t : OUI')
print('Les décalages r(t+1)...r(t+5) correspondent aux lignes futures : OUI')
print(alignment.head(3).to_string())

Tous les rendements constitutifs de RV_5 sont strictement postérieurs à t : OUI
Les décalages r(t+1)...r(t+5) correspondent aux lignes futures : OUI
           date_t_plus_1 date_t_plus_2 date_t_plus_3 date_t_plus_4 date_t_plus_5
date_t                                                                          
1926-07-01    1926-07-02    1926-07-06    1926-07-07    1926-07-08    1926-07-09
1926-07-02    1926-07-06    1926-07-07    1926-07-08    1926-07-09    1926-07-10
1926-07-06    1926-07-07    1926-07-08    1926-07-09    1926-07-10    1926-07-12


## 7. Valeurs manquantes et échantillon final aligné

Les cinq cibles manquantes en fin de fichier sont structurelles : leurs cinq séances futures ne sont pas encore toutes présentes. Elles sont exclues, jamais imputées.

In [7]:
audit = pd.Series({
    'lignes_source': len(factors),
    'dates_dupliquees': int(factors.index.duplicated().sum()),
    'valeurs_brutes_manquantes': int(factors[expected_columns].isna().sum().sum()),
    'rendements_marche_manquants': int(factors['market_return'].isna().sum()),
    'cibles_rv5_manquantes': int(factors['rv_5'].isna().sum()),
    'lignes_finales_alignees': int(factors['rv_5'].notna().sum()),
})
aligned = factors[['market_return', 'rv_5']].dropna().copy()
assert len(aligned) == len(factors) - HORIZON
assert aligned.notna().all().all()
print(audit.to_string())
print(f'Échantillon aligné : {aligned.index.min().date()} → {aligned.index.max().date()}')

lignes_source                  26317
dates_dupliquees                   0
valeurs_brutes_manquantes          0
rendements_marche_manquants        0
cibles_rv5_manquantes              5
lignes_finales_alignees        26312
Échantillon aligné : 1926-07-01 → 2026-08-24


## Conclusion de l'audit

L'audit est réussi si toutes les assertions s'exécutent : source identifiable, unités converties explicitement, dates ordonnées et uniques, calcul manuel reproduit, cinq rendements strictement futurs par cible, et seules les cinq dernières cibles manquantes.

Cette conclusion valide la construction technique de la cible pour ce millésime ; elle ne valide encore ni sa qualité économique comme mesure du risque latent, ni un modèle prédictif.